In [ ]:

import numpy as np
import pandas as pd

#các thư viện liên quan đến sklearn
from sklearn.model_selection import train_test_split    #chia test
from sklearn.linear_model import LogisticRegression #mô hình hồi quy logistic
from sklearn.naive_bayes import MultinomialNB #mô hình multinomial naive bayes
from sklearn.feature_extraction.text import CountVectorizer #véc tơ hóa dữ liệu
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

#các thư viện liên quan đến preprocessing văn bản
import re
import nltk
#from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer #stemming
from nltk.stem import WordNetLemmatizer #lemmatizer

# Tải tài nguyên cần thiết cho việc tách từ
#nltk.download('punkt')
#nltk.download('punkt_tab')

#Tải tài nguyên stopwords
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
data = pd.read_csv('spam_ham_dataset.csv')
data = data.drop_duplicates(subset='text')

In [ ]:
data.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [ ]:
label = data['label_num'].copy()
pre_process_text = data['text'].copy()

pre_process_text.head()

,text
0,Subject: enron methanol ; meter # : 988291\r\n...
1,"Subject: hpl nom for january 9 , 2001\r\n( see..."
2,"Subject: neon retreat\r\nho ho ho , we ' re ar..."
3,"Subject: photoshop , windows , office . cheap ..."
4,Subject: re : indian springs\r\nthis deal is t...


In [ ]:
stop_words = set(stopwords.words('english'))
stemming = PorterStemmer()
def clean_text(text):
    #ta làm sạch dữ liệu bằng cách tạo biến text
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r'[^a-zA-Z]', " ", text)
    #text đã là 1 string sạch nên giờ ta token hóa
    text = text.split()

    #loại bỏ stopword
    text = [word for word in text if word not in stop_words]

    #stemming để đưa về dạng chuẩn của từ(có thể cho ra những từ không thực tế)
    text = [stemming.stem(word) for word in text]

    #đưa lại về dạng string để sử dụng BoW
    text = " ".join(text)
    return text

pre_process_text = pre_process_text.copy().apply(clean_text)

In [ ]:
clean_data = pd.DataFrame(
    {
        'text': pre_process_text,
        'label': label
    }
)


In [ ]:
#phân chia dữ liệu
x = clean_data['text']
y = clean_data['label']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,
                                                    stratify=clean_data["label"], random_state=2910)


In [ ]:
#véc tơ hóa dữ liệu, sử dụng bag of words
vectorizer = CountVectorizer()
# Sử dụng biến mới để tránh ghi đè lên x_train, x_test gốc
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

model_NB = MultinomialNB()
model_NB.fit(x_train_vec, y_train)

model_LR = LogisticRegression()
model_LR.fit(x_train_vec, y_train)

LogisticRegression()

In [ ]:
y_pred_NB = model_NB.predict(x_test_vec)
y_pred_LR = model_LR.predict(x_test_vec)


In [ ]:
#kết quả của model Naive Bayes Multinomial
accuracy_NB = accuracy_score(y_test, y_pred_NB)
precision_NB = precision_score(y_test, y_pred_NB)
recall_NB = recall_score(y_test, y_pred_NB)
confusion_matrix_NB = confusion_matrix(y_test, y_pred_NB)

#kết quả của model Logistic Regression
accuracy_LR = accuracy_score(y_test, y_pred_LR)
precision_LR = precision_score(y_test, y_pred_LR)
recall_LR = recall_score(y_test, y_pred_LR)
confusion_matrix_LR = confusion_matrix(y_test, y_pred_LR)

In [ ]:
#biểu diễn các kết quả
print("Naive Bayes Multinomial:")
print("Accuracy:", accuracy_NB)
print("Precision:", precision_NB)
print("Recall:", recall_NB)
print("Confusion Matrix:\n", confusion_matrix_NB)

print("\nLogistic Regression:")
print("Accuracy:", accuracy_LR)
print("Precision:", precision_LR)
print("Recall:", recall_LR)
print("Confusion Matrix:\n", confusion_matrix_LR)

Naive Bayes Multinomial:
Accuracy: 0.9719719719719719
Precision: 0.9616724738675958
Recall: 0.9419795221843004
Confusion Matrix:
 [[695  11]
 [ 17 276]]

Logistic Regression:
Accuracy: 0.980980980980981
Precision: 0.9659863945578231
Recall: 0.9692832764505119
Confusion Matrix:
 [[696  10]
 [  9 284]]


In [273]:
def predict_spam(text, model):
    text = clean_text(text)
    text_vec = vectorizer.transform([text])
    prediction = model.predict(text_vec)
    if prediction[0] == 1:
        return "Spam"
    else:
        return "Not Spam"

import joblib
joblib.dump(vectorizer, "vectorizer.joblib")
joblib.dump(model_LR, "model_lr.joblib")

['model_lr.joblib']